In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from pymorphy2 import MorphAnalyzer
import numpy as np
from string import punctuation
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim
from gensim.models import Word2Vec
import ast
from gensim.models import FastText

In [2]:
negative_file='D:\\NLP\\lab1\\negative.csv'
positive_file='D:\\NLP\\lab1\\positive.csv'

In [3]:
negative_df = pd.read_csv(negative_file, delimiter=";",header=None, names=["id", "tdate", "tmane",'ttext', 'ttype', 'trep', 'trtv','tfav', 'tstcount', 'tfol', 'tfrien', 'listcount'])
positive_df = pd.read_csv(positive_file, delimiter=";",header=None, names=["id", "tdate", "tmane",'ttext', 'ttype', 'trep', 'trtv','tfav', 'tstcount', 'tfol', 'tfrien', 'listcount'])

In [4]:
df = pd.concat([negative_df, positive_df]).reset_index(drop=True)
df.head()

,id,tdate,tmane,ttext,ttype,trep,trtv,tfav,tstcount,tfol,tfrien,listcount
0,408906762813579264,1386325944,dugarchikbellko,на работе был полный пиддес :| и так каждое за...,-1,0,0,0,8064,111,94,2
1,408906818262687744,1386325957,nugemycejela,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,0,0,0,26,42,39,0
2,408906858515398656,1386325966,4post21,@elina_4post как говорят обещаного три года жд...,-1,0,0,0,718,49,249,0
3,408906914437685248,1386325980,Poliwake,"Желаю хорошего полёта и удачной посадки,я буду...",-1,0,0,0,10628,207,200,0
4,408906914723295232,1386325980,capyvixowe,"Обновил за каким-то лешим surf, теперь не рабо...",-1,0,0,0,35,17,34,0


In [5]:
df = df[['ttext', 'ttype']]
df['ttype'] = df['ttype'].apply(lambda x: 0 if x < 0 else 1)
df.head()

,ttext,ttype
0,на работе был полный пиддес :| и так каждое за...,0
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",0
2,@elina_4post как говорят обещаного три года жд...,0
3,"Желаю хорошего полёта и удачной посадки,я буду...",0
4,"Обновил за каким-то лешим surf, теперь не рабо...",0


In [6]:
nltk.download("stopwords")
nltk.download('punkt_tab')
stop_words = stopwords.words("russian")
punctuations = list(punctuation)
punkt = ['``','...',"''",'«','»','…','”','”','“','-','–','..',':','|','@']
punctuations.extend(punkt)
morph = MorphAnalyzer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lezchook\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lezchook\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [7]:
print(stop_words)

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

In [8]:
remove_tokens = ['не', 'нельзя', 'лучше', 'хорошо', 'никогда', 'зачем', 'нет', 'да']
stop_words = [x for x in stop_words if x not in remove_tokens]

In [9]:
def text_preprocessing(text):
    tokens = nltk.word_tokenize(text, language="russian")
    tokens = [word for word in tokens if word.lower() not in stop_words and word not in punctuations]
    lemmas = [morph.parse(token)[0].normal_form for token in tokens]
    return lemmas

df['ttext'] = df['ttext'].apply(text_preprocessing)
df.head()

,ttext,ttype
0,"[работа, полный, пиддес, каждый, закрытие, мес...",0
1,"[коллега, сидеть, рубиться, urban, terror, из-...",0
2,"[elina_4post, говорить, обещаной, год, ждать]",0
3,"[желать, хороший, полёт, удачный, посадка, быт...",0
4,"[обновить, какой-то, леший, surf, не, работать...",0


In [10]:
df.to_csv('df.csv')

In [11]:
linis_crowd_dict = pd.read_csv('words_all_full_rating.csv', delimiter=";", encoding="windows-1251")
linis_crowd_dict = linis_crowd_dict.drop(linis_crowd_dict.columns[-1], axis=1)

linis_crowd_dict.head()

,Words,mean,dispersion,average rate
0,аборигенный,"-0,25","0,433012701892219",0
1,аборт,-1,"0,816496580927726",-1
2,абрамович,0,0,0
3,абсолютный,"0,333333333333333","0,471404520791032",0
4,абстрактный,"-0,111111111111111","0,87488976377909",0


In [12]:
linis_crowd_dict = dict(zip(linis_crowd_dict['Words'], linis_crowd_dict['mean']))

def feature_transform(tokens):

    result = [float(str(linis_crowd_dict[token]).replace(',', '.')) for token in tokens if token in linis_crowd_dict]

    if len(result) == 0:
        result = [0]

    result = np.array(result)
    result = {
        'mean' : result.mean(), 
        'max' : result.max(), 
        'min' : result.min(), 
        'sum' : result.sum(), 
        'positive' : np.sum(result > 0), 
        'negative' : np.sum(result < 0)
    }
    return result    

In [13]:
df_linis_crowd = pd.concat([df, df['ttext'].apply(feature_transform).apply(pd.Series)], axis=1)
df_linis_crowd.head()

,ttext,ttype,mean,max,min,sum,positive,negative
0,"[работа, полный, пиддес, каждый, закрытие, мес...",0,0.129630,0.222222,0.0,0.388889,2.0,0.0
1,"[коллега, сидеть, рубиться, urban, terror, из-...",0,-0.200000,-0.200000,-0.2,-0.200000,0.0,1.0
2,"[elina_4post, говорить, обещаной, год, ждать]",0,0.000000,0.000000,0.0,0.000000,0.0,0.0
3,"[желать, хороший, полёт, удачный, посадка, быт...",0,0.740741,1.222222,0.0,2.222222,2.0,0.0
4,"[обновить, какой-то, леший, surf, не, работать...",0,0.000000,0.000000,0.0,0.000000,0.0,0.0


In [14]:
def morphy_analyzer(tokens):
    counter = Counter()
    total = 0
    for token in tokens:
        pos = morph.parse(token)[0].tag.POS
        counter[pos] += 1
        total += 1

    if total > 0:
        return {pos : count / total for pos, count in counter.items() if pos is not None}
    else:
        return {}

In [15]:
df_morphy = pd.concat([df, df['ttext'].apply(morphy_analyzer).apply(pd.Series)], axis=1)
df_morphy = df_morphy.fillna(0)
df_morphy.head()

,ttext,ttype,NOUN,ADJF,INFN,PREP,PRCL,ADVB,NPRO,INTJ,PRED,CONJ,NUMR,VERB,ADJS,PRTS,GRND,PRTF,COMP
0,"[работа, полный, пиддес, каждый, закрытие, мес...",0,0.500000,0.250000,0.125000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,"[коллега, сидеть, рубиться, urban, terror, из-...",0,0.300000,0.000000,0.300000,0.1,0.100000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,"[elina_4post, говорить, обещаной, год, ждать]",0,0.200000,0.200000,0.400000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,"[желать, хороший, полёт, удачный, посадка, быт...",0,0.181818,0.181818,0.272727,0.0,0.000000,0.181818,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,"[обновить, какой-то, леший, surf, не, работать...",0,0.285714,0.142857,0.285714,0.0,0.142857,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
def preprocess_tfidf(tokens):
    return ' '.join(tokens)

stop_words_tfidf = ['00', '10', '100', '11', '12', '13', '14', '15', '16', '17', '18', '20', '2013', '2014', '21', '25', '30', '33', '40', '50', '99', 'http', 'https', 'follow', 'amp', 'cio_optimal', 'co', 'dd', 'ddd', 'gt', 'rt', 'the', 'lt', 'mtvstars', 'teamfollowback', 'tukvasociopat']
vectorizer = TfidfVectorizer(max_features=1000, stop_words=stop_words_tfidf)
X_tfidf = vectorizer.fit_transform(df["ttext"].apply(preprocess_tfidf))
df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
df_tfidf = pd.concat([df, df_tfidf], axis=1)
df_tfidf.head()

In [26]:
df_linis_crowd.to_csv('df_linis_crowd.csv')
df_morphy.to_csv('df_morphy.csv')
df_tfidf.to_csv('df_tfidf.csv')

In [27]:
my_w2v_model = Word2Vec(sentences=df['ttext'], vector_size=300, window=5, min_count=1, workers=8)

In [28]:
def preprocess_my_word2vec(tokens):
    result = [my_w2v_model.wv[token] for token in tokens]
    if result:
        return np.array(result).mean(axis=0)
    else:
        return np.zeros(my_w2v_model.vector_size)

df_my_word2vec = pd.DataFrame(df['ttext'].apply(preprocess_my_word2vec).to_list(), columns=np.arange(1, 301))
df_my_word2vec = pd.concat([df, df_my_word2vec], axis=1)
df_my_word2vec.head()

,ttext,ttype,1,2,3,4,5,6,7,8,...,291,292,293,294,295,296,297,298,299,300
0,"[работа, полный, пиддес, каждый, закрытие, мес...",0,0.039631,0.219668,-0.085444,0.083814,-0.285673,-0.572429,0.358411,1.055888,...,0.373322,0.256666,0.458109,-0.230062,0.674155,0.854625,0.040779,-0.229935,0.620862,0.031448
1,"[коллега, сидеть, рубиться, urban, terror, из-...",0,-0.038377,0.366137,0.107436,0.318463,0.010959,-0.563412,0.333739,0.739853,...,0.121193,0.212101,0.744527,0.143959,0.443340,0.864757,0.167241,-0.174059,0.705472,-0.188346
2,"[elina_4post, говорить, обещаной, год, ждать]",0,-0.232475,-0.007386,-0.203224,0.282382,-0.433009,-0.553842,0.405757,1.076935,...,0.108204,0.479133,0.273070,-0.459904,0.699584,0.774611,-0.307445,-0.215409,0.553569,0.069836
3,"[желать, хороший, полёт, удачный, посадка, быт...",0,0.171506,0.325217,-0.219056,0.198447,-0.288000,-0.715441,0.450623,1.058999,...,-0.162712,0.389086,0.321136,0.056675,0.574195,1.254429,0.043492,0.106892,0.718014,-0.259205
4,"[обновить, какой-то, леший, surf, не, работать...",0,-0.004795,0.224696,0.062521,0.382201,-0.315579,-0.489046,0.270137,0.865243,...,0.067688,0.260892,0.690330,0.096212,0.535675,0.585766,0.206344,-0.110451,0.671261,-0.211495


In [29]:
df_my_word2vec.to_csv('df_my_word2vec.csv')

In [30]:
w2v_model = gensim.models.KeyedVectors.load_word2vec_format('model.bin', encoding='utf-8', unicode_errors='ignore', binary=True)

In [31]:
df_pos = pd.read_csv('df_pos.csv', delimiter=",").drop(['Unnamed: 0.1',	'Unnamed: 0'], axis=1)
df_pos.head()

,ttext,ttype,ttext_pos
0,"['работа', 'полный', 'пиддес', 'каждый', 'закр...",0,"['работа_NOUN', 'полный_ADJ', 'пиддосить_NOUN'..."
1,"['коллега', 'сидеть', 'рубиться', 'urban', 'te...",0,"['коллега_NOUN', 'сидеть_VERB', 'рубиться_VERB..."
2,"['elina_4post', 'говорить', 'обещаной', 'год',...",0,"['elina4post_X', 'говорить_VERB', 'обещаный_NO..."
3,"['желать', 'хороший', 'полёт', 'удачный', 'пос...",0,"['желать_VERB', 'хороший_ADJ', 'полет_VERB', '..."
4,"['обновить', 'какой-то', 'леший', 'surf', 'раб...",0,"['обновлять_VERB', 'какой-то_ADJ', 'леший_ADJ'..."


In [32]:
def preprocess_word2vec(tokens):
    tokens = ast.literal_eval(tokens)
    mean = []

    for token in tokens:
        if token in w2v_model.key_to_index.keys():
            mean.append(w2v_model[w2v_model.key_to_index[token]])

    if not mean:
        return np.zeros(w2v_model.vector_size)

    mean = gensim.matutils.unitvec(np.array(mean).mean(axis=0)).astype(np.float32)
    return mean

df_word2vec = pd.DataFrame(df_pos['ttext_pos'].apply(preprocess_word2vec).to_list(), columns=np.arange(1, 301))
df_word2vec = pd.concat([df, df_word2vec], axis=1)
df_word2vec.head()

,ttext,ttype,1,2,3,4,5,6,7,8,...,291,292,293,294,295,296,297,298,299,300
0,"[работа, полный, пиддес, каждый, закрытие, мес...",0,-0.025578,0.098892,0.100078,0.043680,0.054846,0.022513,0.026592,-0.000200,...,-0.090991,-0.067886,-0.006360,-0.055676,-0.001083,0.088391,0.022902,0.048532,0.034008,0.128153
1,"[коллега, сидеть, рубиться, urban, terror, из-...",0,0.018884,0.161164,0.023477,-0.024357,0.041295,-0.044964,-0.049446,-0.014660,...,-0.008218,-0.075631,0.067400,-0.033227,-0.024125,-0.015628,-0.022084,-0.057485,0.005965,-0.056475
2,"[elina_4post, говорить, обещаной, год, ждать]",0,0.035387,0.041662,0.029343,-0.010137,-0.063307,-0.008787,0.003457,-0.023158,...,0.032419,-0.008152,0.031810,-0.037580,0.079544,0.038071,-0.067372,-0.003743,0.052041,0.076392
3,"[желать, хороший, полёт, удачный, посадка, быт...",0,-0.039753,0.183311,0.048805,0.008788,-0.007898,-0.039873,-0.005032,0.033402,...,0.010154,0.042666,-0.037772,0.001342,0.009754,0.023223,0.100067,-0.032186,0.006097,0.051124
4,"[обновить, какой-то, леший, surf, не, работать...",0,-0.054380,0.101400,0.109294,0.073208,-0.036111,-0.050060,0.046296,0.081363,...,0.062744,0.000255,0.038924,0.064392,-0.069846,0.036937,-0.112130,0.032484,0.061183,0.023959


In [33]:
df_word2vec.to_csv('df_word2vec.csv')

In [34]:
my_fastText_model = FastText(sentences=df['ttext'], vector_size=300, window=5, min_count=1, workers=8)

In [35]:
def preprocess_my_fastText(tokens):
    result = [my_fastText_model.wv[token] for token in tokens]
    if result:
        return np.array(result).mean(axis=0)
    else:
        return np.zeros(my_fastText_model.vector_size)

df_my_fastText = pd.DataFrame(df['ttext'].apply(preprocess_my_fastText).to_list(), columns=np.arange(1, 301))
df_my_fastText = pd.concat([df, df_my_fastText], axis=1)
df_my_fastText.head()

,ttext,ttype,1,2,3,4,5,6,7,8,...,291,292,293,294,295,296,297,298,299,300
0,"[работа, полный, пиддес, каждый, закрытие, мес...",0,0.015538,-0.056934,0.060281,0.063772,-0.217311,-0.400256,0.103396,0.098359,...,0.358993,0.147403,-0.604583,-0.032806,-0.351147,-0.332578,0.635437,0.535938,0.733109,0.355754
1,"[коллега, сидеть, рубиться, urban, terror, из-...",0,-0.557949,-0.142219,0.541082,0.207650,0.296615,-0.047279,-0.094600,-0.134059,...,0.277087,0.043083,-0.482085,0.038894,-0.417339,-0.531212,0.666107,0.404946,1.144669,0.608411
2,"[elina_4post, говорить, обещаной, год, ждать]",0,-0.287099,-0.069515,0.347607,-0.367344,-1.023940,-0.255836,0.061885,0.270523,...,-0.293794,-0.008569,-1.039471,-0.054548,-0.183832,-0.471962,-0.340224,0.157353,0.791440,0.913336
3,"[желать, хороший, полёт, удачный, посадка, быт...",0,-0.172419,0.306317,0.058805,-0.337325,-0.080656,-0.461286,0.318689,0.279804,...,0.426579,0.293990,-1.136548,0.114569,-0.300089,-0.627816,0.116387,0.620176,0.632405,0.812193
4,"[обновить, какой-то, леший, surf, не, работать...",0,-0.083643,-0.048478,0.539883,0.259408,-0.110939,-0.138205,0.572468,-0.065460,...,0.378510,0.226278,-0.397049,-0.094855,-0.532697,-1.016661,0.401528,0.435306,1.230254,0.802640


In [36]:
df_my_fastText.to_csv('df_my_fastText.csv')

In [37]:
fastText_model = gensim.models.fasttext.FastTextKeyedVectors.load('model.model')

In [38]:
def preprocess_fastText(tokens):
    result = [fastText_model.get_vector(token) for token in tokens]
    if result:
        return np.array(result).mean(axis=0)
    else:
        return np.zeros(fastText_model.vector_size)

df_fastText = pd.DataFrame(df['ttext'].apply(preprocess_fastText).to_list(), columns=np.arange(1, 301))
df_fastText = pd.concat([df, df_fastText], axis=1)
df_fastText.head()

,ttext,ttype,1,2,3,4,5,6,7,8,...,291,292,293,294,295,296,297,298,299,300
0,"[работа, полный, пиддес, каждый, закрытие, мес...",0,-0.021080,0.058871,-0.028595,0.204365,0.099959,0.192865,0.003974,0.098822,...,0.032653,0.130866,0.088766,-0.045322,-0.161956,0.052231,0.061403,-0.098500,-0.248108,-0.003967
1,"[коллега, сидеть, рубиться, urban, terror, из-...",0,0.077991,0.095402,-0.010929,0.092279,0.084164,0.018748,0.150167,-0.045739,...,0.134403,0.097092,0.090820,0.033078,-0.173603,0.116264,0.049419,-0.003961,-0.091165,-0.013591
2,"[elina_4post, говорить, обещаной, год, ждать]",0,-0.061520,0.018157,-0.130235,-0.021144,0.208054,0.213792,0.023104,0.051409,...,0.025983,-0.021074,0.116292,0.081605,-0.104921,-0.025903,0.120133,0.043922,-0.120371,0.002481
3,"[желать, хороший, полёт, удачный, посадка, быт...",0,-0.023305,0.010863,-0.038696,0.349210,0.161522,0.060291,0.008188,-0.057161,...,-0.052738,0.084836,0.156556,0.023954,-0.139426,0.042694,0.126957,-0.115903,-0.072198,0.109510
4,"[обновить, какой-то, леший, surf, не, работать...",0,0.062192,0.150368,-0.084928,0.098968,0.117249,0.040068,0.083630,0.044029,...,0.045102,0.117518,0.131550,0.139264,-0.167963,0.086855,0.171978,-0.162369,-0.123973,0.048142


In [39]:
df_fastText.to_csv('df_fastText.csv')